In [53]:
from pathlib import Path

INPUT_CUSTOMER_CATALOGUE = Path(r'C:\Users\reyiv\Documents\Quaker Houghton\Codigos\limpieza-brandname rep\limpieza-brandname\Fuzzy Cleaning\Industrial_Market.csv')
INPUT_EXACT_GROUPS = Path(r'C:\Users\reyiv\Documents\Quaker Houghton\Codigos\limpieza-brandname rep\limpieza-brandname\Fuzzy Cleaning\output\02_exact_normalized_groups.csv')
INPUT_REVIEWED_MATCHES = Path(r'C:\Users\reyiv\Documents\Quaker Houghton\Codigos\limpieza-brandname rep\limpieza-brandname\Fuzzy Cleaning\output\04_fuzzy_matches_reviewed.csv')

OUTPUT_DIR = Path(r'C:\Users\reyiv\Documents\Quaker Houghton\Codigos\limpieza-brandname rep\limpieza-brandname\Fuzzy Cleaning\output')
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_GP_MAPPING = OUTPUT_DIR / "05_gp_mapping.csv"
OUTPUT_CUSTOMER_CATALOGUE = OUTPUT_DIR / "Customer Catalogue Clean.csv"
OUTPUT_REPORT = OUTPUT_DIR / "Cleaning Report.txt"

DECISION_NO = "NO"

In [54]:
from collections import defaultdict
from pathlib import Path
import ast

import pandas as pd

In [55]:
print("Loading input files...")

df_catalogue = pd.read_csv(INPUT_CUSTOMER_CATALOGUE)
df_exact = pd.read_csv(INPUT_EXACT_GROUPS)
df_review = pd.read_csv(INPUT_REVIEWED_MATCHES)

print(f"Customer catalogue: {len(df_catalogue):,} rows")
print(f"Exact normalization groups: {len(df_exact):,}")
print(f"Reviewed fuzzy matches: {len(df_review):,}")

Loading input files...
Customer catalogue: 49,493 rows
Exact normalization groups: 561
Reviewed fuzzy matches: 711


In [56]:
CATALOGUE_COLUMNS = {
    "Global Parent"
}

EXACT_COLUMNS = {
    "Normalized_GP",
    "Variants"
}

REVIEW_COLUMNS = {
    "Representative_1",
    "Representative_2",
    "Decision"
}


def validate_columns(df, required_columns, dataset_name):

    missing = required_columns - set(df.columns)

    if missing:

        raise ValueError(
            f"{dataset_name} is missing columns: {sorted(missing)}"
        )



validate_columns(
    df_catalogue,
    CATALOGUE_COLUMNS,
    "Customer Catalogue"
)

validate_columns(
    df_exact,
    EXACT_COLUMNS,
    "Exact Groups"
)

validate_columns(
    df_review,
    REVIEW_COLUMNS,
    "Reviewed Matches"
)

print("All input files validated successfully.")

All input files validated successfully.


In [57]:
df_review = df_review.copy()

df_review["Decision"] = (
    df_review["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)

approved_merges = df_review[
    (df_review["Decision"] != "")
    &
    (df_review["Decision"].str.upper() != DECISION_NO)
].copy()

print(f"Approved manual homologations: {len(approved_merges):,}")

Approved manual homologations: 380


In [58]:
print("Building exact normalization mapping...")

gp_mapping = {}

for _, row in df_exact.iterrows():

    canonical_gp = row["Normalized_GP"]

    variants = ast.literal_eval(row["Variants"])

    for variant in variants:

        if (
            variant in gp_mapping
            and
            gp_mapping[variant] != canonical_gp
        ):

            raise ValueError(
                f"Conflicting exact mapping for '{variant}'"
            )

        gp_mapping[variant] = canonical_gp

print(f"Exact mappings created: {len(gp_mapping):,}")

Building exact normalization mapping...
Exact mappings created: 1,158


In [59]:
print("\nFirst 10 mappings:\n")

for i, (original, canonical) in enumerate(gp_mapping.items()):

    print(f"{original}  -->  {canonical}")

    if i == 9:
        break


First 10 mappings:

GENUINE PARTS  -->  GENUINE PARTS
GENUINE PARTS COMPANY  -->  GENUINE PARTS
PENTAIR  -->  PENTAIR
Pentair  -->  PENTAIR
MOLEX INC  -->  MOLEX
MOLEX  -->  MOLEX
TENNECO  -->  TENNECO
Tenneco  -->  TENNECO
ALFA LAVAL  -->  ALFA LAVAL
ALFA LAVAL INC  -->  ALFA LAVAL


In [60]:
# ============================================================
# Section 8 - Build Homologation Graph
# ============================================================

graph = defaultdict(set)


print("Adding exact normalization edges...")

for _, row in df_exact.iterrows():

    variants = ast.literal_eval(row["Variants"])

    if len(variants) < 2:
        continue

    first = variants[0]

    for variant in variants[1:]:

        graph[first].add(variant)
        graph[variant].add(first)

print("Exact normalization edges added.")




print("Adding manual homologation edges...")

for _, row in approved_merges.iterrows():

    gp1 = row["Representative_1"]
    gp2 = row["Representative_2"]

    graph[gp1].add(gp2)
    graph[gp2].add(gp1)

print("Manual homologation edges added.")

Adding exact normalization edges...
Exact normalization edges added.
Adding manual homologation edges...
Manual homologation edges added.


In [61]:
visited = set()

components = []


def dfs(start):

    stack = [start]

    component = []

    while stack:

        node = stack.pop()

        if node in visited:
            continue

        visited.add(node)

        component.append(node)

        stack.extend(graph[node] - visited)

    return component


for node in graph:

    if node not in visited:

        components.append(dfs(node))

print(f"Connected components found: {len(components):,}")

Connected components found: 868


In [62]:
# ============================================================
# Section 10 - Validate Canonical Decisions
# ============================================================

decision_lookup = {}

for _, row in approved_merges.iterrows():

    gp1 = row["Representative_1"]
    gp2 = row["Representative_2"]
    decision = row["Decision"]

    decision_lookup[frozenset([gp1, gp2])] = decision

component_canonicals = {}

conflicts = []

for component in components:

    component_set = set(component)

    decisions = set()

    conflicting_rows = []

    for _, row in approved_merges.iterrows():

        gp1 = row["Representative_1"]
        gp2 = row["Representative_2"]

        if gp1 in component_set and gp2 in component_set:

            decisions.add(row["Decision"])

            conflicting_rows.append({
                "Representative_1": gp1,
                "Representative_2": gp2,
                "Decision": row["Decision"]
            })

    if len(decisions) > 1:

        conflicts.append({
            "Component": sorted(component),
            "Canonicals": sorted(decisions),
            "Rows": conflicting_rows
        })

    elif len(decisions) == 1:

        component_canonicals[frozenset(component)] = next(iter(decisions))

    else:

        canonical = None

        for gp in component:

            mapped = gp_mapping.get(gp)

            if mapped in component:

                canonical = mapped
                break

        if canonical is None:
            canonical = min(component)

        component_canonicals[frozenset(component)] = canonical

if conflicts:

    print(f"\nFound {len(conflicts)} conflicting connected components.\n")

    for i, conflict in enumerate(conflicts, start=1):

        print("=" * 80)
        print(f"Conflict #{i}")
        print("=" * 80)

        print("\nComponent:")

        for gp in conflict["Component"]:
            print(f"  - {gp}")

        print("\nDifferent canonical decisions:")

        for decision in conflict["Canonicals"]:
            print(f"  - {decision}")

        print("\nRows causing the conflict:")

        for row in conflict["Rows"]:

            print(
                f"  {row['Representative_1']}  <-->  "
                f"{row['Representative_2']}  "
                f"=>  {row['Decision']}"
            )

        print()

    raise ValueError(
        f"{len(conflicts)} conflicting connected components were found."
    )

print("Canonical validation passed.")

Canonical validation passed.


In [63]:
print("Updating GP mapping...")

for component, canonical in component_canonicals.items():

    for gp in component:

        gp_mapping[gp] = canonical

print(f"Final GP mappings: {len(gp_mapping):,}")

Updating GP mapping...
Final GP mappings: 1,838


In [64]:
# ============================================================
# Section 12 - Apply GP Mapping
# ============================================================

print("Applying GP mapping...")

df_clean = df_catalogue.copy()

df_clean["Global Parent"] = (
    df_clean["Global Parent"]
    .map(gp_mapping)
    .fillna(df_clean["Global Parent"])
)

print("GP mapping applied.")

Applying GP mapping...
GP mapping applied.


In [65]:
# ============================================================
# Section 13 - Validate Output
# ============================================================

assert len(df_clean) == len(df_catalogue)

print("Row count preserved.")

original_unique = df_catalogue["Global Parent"].nunique()

clean_unique = df_clean["Global Parent"].nunique()

print(f"Unique GPs before: {original_unique:,}")
print(f"Unique GPs after : {clean_unique:,}")
print(f"Reduction         : {original_unique - clean_unique:,}")

unmapped = [
    gp
    for gp in gp_mapping
    if gp_mapping[gp] not in set(df_clean["Global Parent"])
]

print(f"Unused mappings: {len(unmapped)}")

Row count preserved.
Unique GPs before: 18,780
Unique GPs after : 17,987
Reduction         : 793
Unused mappings: 61


In [66]:
# ============================================================
# Section 14 - Exports
# ============================================================


df_clean.to_csv(
    OUTPUT_CUSTOMER_CATALOGUE,
    index=False
)

mapping_df = (
    pd.DataFrame(
        gp_mapping.items(),
        columns=[
            "Original_GP",
            "Canonical_GP"
        ]
    )
    .sort_values("Original_GP")
)

mapping_df.to_csv(
    OUTPUT_GP_MAPPING,
    index=False
)

print(f"GP mapping exported to '{OUTPUT_GP_MAPPING}'.")

GP mapping exported to 'C:\Users\reyiv\Documents\Quaker Houghton\Codigos\limpieza-brandname rep\limpieza-brandname\Fuzzy Cleaning\output\05_gp_mapping.csv'.


In [67]:
changes = df_catalogue.copy()

changes["Global Parent Clean"] = df_clean["Global Parent"]

changes = changes[
    changes["Global Parent"] != changes["Global Parent Clean"]
]

changes = changes[
    [
        "Global Parent",
        "Global Parent Clean",
        "Customer Name",
        "Ship To Country",
        "Ship To State",
        "Ship To City"
    ]
]

In [68]:
# ============================================================
# Section 17 - Generate Cleaning Report
# ============================================================

report = f"""
============================================================
          GLOBAL PARENT CLEANING REPORT
============================================================

Execution Summary
-----------------
Rows processed                 : {len(df_catalogue):,}

Unique Global Parents
---------------------
Original unique GPs            : {df_catalogue['Global Parent'].nunique():,}
Clean unique GPs               : {df_clean['Global Parent'].nunique():,}
Reduction in unique GPs        : {df_catalogue['Global Parent'].nunique() - df_clean['Global Parent'].nunique():,}

Exact Normalization
-------------------
Exact normalization groups     : {len(df_exact):,}
Names in GP mapping            : {len(gp_mapping):,}

Manual Homologation
-------------------
Reviewed candidate pairs       : {len(df_review):,}
Approved homologations         : {len(approved_merges):,}

Graph Analysis
--------------
Connected components           : {len(components):,}

Catalogue Changes
-----------------
Rows modified                  : {len(changes):,}
Rows unchanged                 : {len(df_catalogue) - len(changes):,}

Output Files
------------
- {OUTPUT_CUSTOMER_CATALOGUE.name}
- {OUTPUT_GP_MAPPING.name}

Status
------
Cleaning completed successfully.
No conflicting canonical decisions were detected.

============================================================
"""

with open(OUTPUT_REPORT, "w", encoding="utf-8") as f:
    f.write(report)

print(f"Cleaning report exported to '{OUTPUT_REPORT}'.")

Cleaning report exported to 'C:\Users\reyiv\Documents\Quaker Houghton\Codigos\limpieza-brandname rep\limpieza-brandname\Fuzzy Cleaning\output\Cleaning Report.txt'.
